In [ ]:
from google.cloud import bigquery

client = bigquery.Client()
query = """
    SELECT * FROM `numeric-advice-452700-j9.neo_bank_.churn`
"""
df_churn = client.query(query).to_dataframe()

df_churn

In [ ]:
import pandas as pd
import plotly.express as px

# Asegúrate de filtrar usuarios que fueron notificados (has_notification=True)
df_filtrado = df_churn[df_churn['has_notification'] == True].copy()

# Agrupa por canal y estado de conversión
df_grouped = (
    df_filtrado
    .groupby(['channel', 'converted'])['user_id']
    .nunique()
    .reset_index()
    .pivot(index='channel', columns='converted', values='user_id')
    .fillna(0)
    .reset_index()
)

# Renombra columnas para mayor claridad
df_grouped.columns = ['channel', 'No_convirtieron', 'Convirtieron']

# Calcula el total por canal para ordenar
df_grouped['Total'] = df_grouped['No_convirtieron'] + df_grouped['Convirtieron']
df_grouped = df_grouped.sort_values('Total', ascending=False)

display(df_grouped)

# Gráfico de barras apiladas
fig = px.bar(
    df_grouped,
    x='channel',
    y=['No_convirtieron', 'Convirtieron'],
    labels={'value': 'Usuarios', 'variable': 'Estado'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set1,
    text_auto=True,
    title='Usuarios notificados: Convirtieron vs No Convirtieron por Canal'
)

fig.update_layout(yaxis_title='Número de usuarios')
fig.show()


In [ ]:
import plotly.express as px

# Filtrar usuarios activos e inactivos
usuarios_activos = df_churn[df_churn['has_transaction'] == True]
usuarios_inactivos = df_churn[df_churn['has_transaction'] == False]

# Agrupar activos por canal
activos = (
    usuarios_activos.groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_activos')
)

# Agrupar inactivos por canal
inactivos = (
    usuarios_inactivos.groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_inactivos')
)

# Unir ambos
comparacion = activos.merge(inactivos, on='channel', how='outer').fillna(0)

# Calcular total y ordenar por mayor total
comparacion['total'] = comparacion['usuarios_activos'] + comparacion['usuarios_inactivos']
comparacion = comparacion.sort_values('total', ascending=False)

# Mostrar tabla de control
display(comparacion)

# Graficar apilada
fig = px.bar(
    comparacion,
    x='channel',
    y=['usuarios_inactivos', 'usuarios_activos'],
    barmode='stack',
    labels={'value': 'Usuarios', 'variable': 'Estado'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    text_auto=True,
    title='Usuarios Activos vs Inactivos por Canal'
)

fig.update_layout(yaxis_title='Número de usuarios')

# Mostrar gráfica
fig.show()


In [ ]:
# Filtrar usuarios únicos con churn=True
usuarios_churned = (
    df_churn[df_churn['churned'] == True]
    .groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_churned')
)

# Filtrar usuarios únicos con churn=False
usuarios_no_churned = (
    df_churn[df_churn['churned'] == False]
    .groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_churned')
)

# Unir ambos dataframes
comparacion = pd.merge(
    usuarios_churned, usuarios_no_churned,
    on='channel', how='outer'
).fillna(0)

# Calcular total para ordenar
comparacion['total'] = comparacion['usuarios_churned'] + comparacion['usuarios_no_churned']
comparacion = comparacion.sort_values('total', ascending=False)

display(comparacion)

# Graficar barras apiladas
fig = px.bar(
    comparacion,
    x='channel',
    y=['usuarios_churned', 'usuarios_no_churned'],
    barmode='stack',
    labels={'value': 'Usuarios', 'variable': 'Estado'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    text_auto=True,
    title='Usuarios Churned vs No Churned por Canal'
)

fig.update_layout(yaxis_title='Número de usuarios')
fig.show()


In [ ]:
df_churn.groupby('churned')['user_id'].nunique()